# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, examining, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We first list all record sets and their associated fields and columns. Referencing each entity by its unique `@id` ensures consistency.

In [ ]:
# List all record sets and fields with their @id
# The Croissant schema exposes these via dataset.metadata.record_sets

record_sets = dataset.metadata.record_sets

print("Available record sets:")
for rset in record_sets:
    print(f"- Record Set name: {rset.name}")
    print(f"  @id: {rset['@id']}")
    print(f"  Description: {getattr(rset, 'description', '(no description)')}")

    if hasattr(rset, 'fields'):
        print("  Fields:")
        for field in rset.fields:
            print(f"    - Field name: {field.name}")
            print(f"      @id: {field['@id']}")
            print(f"      Data type: {field.data_type}")
    if hasattr(rset, 'columns'):
        print("  Columns:")
        for col in rset.columns:
            print(f"    - Column name: {col.name}")
            print(f"      @id: {col['@id']}")
            print(f"      Data type: {col.data_type}")
    print("-----")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis using their `@id`.

For each record set, we retrieve all records and load them into pandas DataFrames—referencing the record set `@id`.

We will demonstrate with one record set whose `@id` appears in the overview above.

In [ ]:
# Extract data from each record set using their @id

# Collect record set @ids
record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded DataFrame for Record Set @id: {rsid}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())

# Example: Show columns and preview for the first record set
if len(record_set_ids) > 0 and record_set_ids[0] in dataframes:
    print("\nSample columns from first record set:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    print(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, and grouping using fields referenced by their `@id`.

Here, we'll select a numeric field and a group field by their respective `@id` based on the previous overview.

In [ ]:
# Choose a record set and relevant numeric/group fields

# Example using the first record set
record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

df = dataframes.get(record_set_id)

# Find numeric columns (those with dtype 'float' or 'int')
if df is not None:
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_cols) > 0:
        numeric_field_id = numeric_cols[0]  # Using first numeric field found
    else:
        numeric_field_id = df.columns[0]  # fallback

    # Threshold for filtering
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical column
    # Pick the second column for grouping, if exists
    if len(df.columns) > 1:
        group_field_id = df.columns[1]
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field and visualize its relationship to the group field (if applicable).

In [ ]:
# Plot histograms and group comparisons
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in Record Set @id {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group field exists, show a boxplot
    if len(df.columns) > 1:
        group_field_id = df.columns[1]
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've guided you through accessing, exploring, filtering, and visualizing the FAIR^2 dataset using the `mlcroissant` library.

- **Metadata Review:** Dataset metadata, including description and relevant collection details, were displayed.
- **Recordset Overview:** Entities and their `@id`s were listed for reference.
- **Extraction and Processing:** Data were extracted, filtered, normalized, and grouped using specified fields.
- **Visualization:** Key distributions and group comparisons were visualized.

Refer to the dataset documentation for more advanced analyses, and remember to always reference data entities by their `@id` for reproducibility.